# Contribution B — Distribution-Aware Exemplar Allocation

Computes `m_c` proportional to `n_c ** alpha` subject to `sum(m_c) = M`, over the
declared alpha grid, at object level and at image level.

> **Scope, stated up front.** This is the allocation **mathematics only**. It produces
> the exemplar buffers an alpha sweep would use; it does **not** measure their effect.
> The research question — what `alpha` is optimal in OWOD and how that optimum moves
> with tail severity — needs real incremental model updates with PROB retraining and
> per-group forgetting measured across tasks. That remains a separate, documented step:
> [`docs/research_design.md`](../docs/research_design.md) section 8.
>
> There is deliberately no offline proxy for forgetting, because forgetting is a
> property of a trained model, not of a buffer. **Contribution B is not experimentally
> complete.**

Needs no GPU and no detector: it runs on class counts.

In [ ]:
# @title Setup
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = Path("/content/distribution-aware-owod")
    if not REPO.exists():
        subprocess.run(["git","clone","https://github.com/gubiczam/distribution-aware-owod.git",str(REPO)], check=True)
    subprocess.run([sys.executable,"-m","pip","install","--quiet","--editable",f"{REPO}[dev]"], check=True)
else:
    REPO = Path.cwd().parent
os.chdir(REPO)
subprocess.run([sys.executable,"-m","pytest","-q","--no-header","tests/test_memory.py"], check=True)

## 1. The three named strategies

`alpha = 0` is the current standard (uniform), `alpha = 1` is size-proportional and
head-favouring, `alpha < 0` is tail-favouring. Two properties are non-negotiable and
are asserted by the tests above: the budget is conserved **exactly**, and ties break
deterministically so an allocation is reproducible.

Note where the availability cap binds: at `alpha < 0` a rare class is allotted more
exemplars than it has objects, so the cap redistributes the remainder and records which
classes were capped — a capped allocation no longer follows `n_c ** alpha`, and a sweep
that ignored that would mis-attribute the effect.

In [ ]:
# @title Object-level sweep on a long-tailed table
import pandas as pd
from daowod.memory import ALPHA_GRID, allocate

CLASS_COUNTS = {"head": 120, "upper_mid": 60, "mid": 30, "lower_mid": 12, "tail": 6, "rare": 2}
TOTAL_MEMORY = 60

rows = []
for alpha in ALPHA_GRID:
    result = allocate(CLASS_COUNTS, total_memory=TOTAL_MEMORY, alpha=alpha)
    assert sum(result.counts.values()) == TOTAL_MEMORY, "conservation is not optional"
    for name, count in result.counts.items():
        rows.append({"alpha": alpha, "class": name, "objects": CLASS_COUNTS[name],
                     "exemplars": count, "capped": name in result.capped_classes})

table = pd.DataFrame(rows).pivot(index="class", columns="alpha", values="exemplars")
display(table.reindex(CLASS_COUNTS.keys()))
print("every column sums to", TOTAL_MEMORY, ":", (table.sum() == TOTAL_MEMORY).all())

## 2. Image-level allocation

One stored image brings **all** of its objects, so no selection of whole images
generally satisfies every per-class quota. That ambiguity is exactly what the proposal
names when it says an exemplar per class can be read at object level or image level.
The selector is greedy and deterministic, and reports the residual per class rather
than hiding it.

In [ ]:
# @title Image-level: alpha changes which images are stored
from daowod.memory import allocate_images

IMAGE_CLASSES = {
    "img_head_a": ["head", "head"],
    "img_head_b": ["head"],
    "img_head_c": ["head"],
    "img_mixed":  ["head", "tail"],
    "img_tail":   ["tail"],
    "img_rare":   ["rare"],
}
for alpha in (1.0, 0.0, -1.0):
    result = allocate_images(IMAGE_CLASSES, total_memory=3, alpha=alpha)
    print(f"alpha={alpha:+.1f} -> {result.image_ids}")
    print(f"           retained {dict(result.counts)}  shortfall {dict(result.shortfall or {})}")

## 3. Running it as an experiment

The entrypoint refuses to run on invented counts: supply either a `class_name,count`
CSV or real VOC annotations plus a split, so the allocation describes an actual OWOD
task. It writes `allocations.csv` and a manifest that records the explicit non-claim.

In [ ]:
# @title Entrypoint
subprocess.run([sys.executable, "experiments/contribution_b.py", "--help"], check=True)

## 4. What remains for Contribution B

1. Wire `ExemplarAllocation` into a replay-based incremental update. The interface is
   already plain data — per-class counts plus, at image level, the image IDs.
2. Retrain PROB across at least two tasks per `alpha`.
3. Measure **per-group forgetting** (head / medium / tail) and grouped mAP, which is
   what identifies the optimal `alpha`.
4. Repeat at each tail severity, to answer how the optimum moves with tail severity.

Steps 2–4 need GPU time and a fixed incremental protocol. Until they are done, no claim
about optimal `alpha` is supported by this repository.